# 03 · Avaliação: as 60 execuções e o juiz

Roda as 20 perguntas nos 3 sistemas (60 execuções), grava uma linha por execução e depois julga cada resposta com o juiz (Gemini), que tem acesso somente leitura ao mesmo servidor MCP.

| Parte | O que faz | Tempo aproximado |
|---|---|---|
| 1 | Prepara a sessão: recorte, bancos, servidor MCP, Ollama e `qwen3:8b` | ~10 min |
| 2 | Abre a sessão com o servidor MCP, calcula o gabarito e aquece o modelo | ~3 min |
| 3 | **As 60 execuções** (20 perguntas × 3 sistemas), gravadas no Drive a cada execução | ~15 min |
| 4 | **O juiz** julga cada resposta contra o gabarito | ~15 min |
| 5 | Resumo rápido por sistema e registro do ambiente | segundos |

As respostas são gravadas **antes** do julgamento, e o juiz roda depois, separado: as chamadas ao Gemini não entram na latência nem nos recursos medidos dos agentes, e dá para julgar de novo sem rodar os agentes outra vez. A análise completa fica no notebook 04, que só lê os arquivos.

**Antes de rodar:**
1. Runtime **T4 GPU** (Ambiente de execução → Alterar o tipo de ambiente de execução).
2. Na Parte 1, ajuste `DRIVE_DIR` (os 8 CSV do recorte) e `RESULTADOS_DIR` (onde ficam os resultados, no Drive).
3. Crie o secret **`GOOGLE_API_KEY`** no Colab (ícone de chave na barra da esquerda) com a chave do Google AI Studio, e libere o acesso deste notebook a ele.

**Se a sessão cair:** rode de novo as Partes 1 e 2 e depois a 3 ou a 4. As execuções e os julgamentos que já estão no Drive são pulados.

In [ ]:
# @title Parte 1 · Preparar a sessão
import os, subprocess, sys

REPO_URL = "https://github.com/felipeafigueiredo/text-to-sql-and-cypher.git"
REPO_DIR = "/content/text-to-sql-and-cypher"
DRIVE_DIR = "/content/drive/MyDrive/airbnb_recorte"          # pasta do Drive com os 8 CSV do recorte
RESULTADOS_DIR = "/content/drive/MyDrive/airbnb_resultados"  # execuções e julgamentos, gravados a cada linha

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h · %s"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
sys.path.insert(0, REPO_DIR)

from ambiente.preparar import preparar
TEMPOS = preparar(DRIVE_DIR)
os.makedirs(RESULTADOS_DIR, exist_ok=True)

In [ ]:
# @title Parte 2 · Sessão com o servidor MCP, gabarito e aquecimento do modelo
from agentes.comum import Servidor, aquecer
from avaliacao.gabarito import calcular

if "SRV" in globals():
    await SRV.fechar()                          # se a célula rodar de novo, fecha a sessão anterior
SRV = await Servidor().abrir()                 # sobe o server.py por stdio, lista as ferramentas e lê os esquemas
print(f"🔧 {len(SRV.ferramentas)} ferramentas · 📄 resources: {list(SRV.esquemas)}")

GABARITO = calcular()                          # o gabarito sai dos bancos, pelas consultas de referência
print(f"📋 {len(GABARITO)} perguntas com gabarito")

print(f"🔥 aquecimento do modelo (fora das métricas): {await aquecer()} s")

---
## Parte 3 · As 60 execuções

Cada pergunta roda nos 3 sistemas em seguida (pergunta por pergunta, não sistema por sistema): se a GPU ficar mais lenta com o tempo, isso pesa igual nos três, e uma sessão que cai no meio deixa as primeiras perguntas completas nos três.

Cada execução grava, no `RESULTADOS_DIR`:
- **`execucoes.csv`**: uma linha por execução, com a resposta, a checagem automática contra o gabarito, a trajetória (a ferramenta esperada foi chamada?), a rota do roteador (sistema 2), latência, chamadas ao modelo e às ferramentas, tokens de entrada e saída (somados entre todos os agentes), CPU, RAM, GPU e devoluções do verificador (sistema 3);
- **`execucoes.jsonl`**: o registro completo (cada consulta com o resultado, cada chamada ao modelo, os vereditos do verificador e as respostas devolvidas), para o juiz e a revisão manual.

A checagem automática é só uma segunda opinião: o acerto que vale é o do juiz (Parte 4) e da revisão manual.

In [ ]:
# @title Rodar as 60 execuções
from agentes.agente_unico import AgenteUnico
from agentes.roteamento import Roteamento
from agentes.verificador import Verificador
from avaliacao.avaliador import avaliar

SISTEMAS = [AgenteUnico(SRV), Roteamento(SRV), Verificador(SRV)]

def progresso(p, r, l):
    marca = "✅" if l["checagem_ok"] else "❌"
    rota = f" · rota {l['rota']}" if l["rota"] else ""
    dev = f" · {l['devolucoes']} devolução(ões)" if l["devolucoes"] else ""
    erro = f" · 💥 {l['erro']}" if l["erro"] else (" · ⛔ limite de passos" if l["estourou_limite"] else "")
    print(f"{marca} {l['execucao']:<7} {l['segundos']:>6.1f} s · {l['chamadas_modelo']:>2} chamadas · "
          f"{l['tokens_total']:>6} tokens{rota}{dev}{erro}")

n = await avaliar(SISTEMAS, GABARITO, RESULTADOS_DIR, mostrar=progresso)
print(f"\n{n} execuções agora · arquivos em {RESULTADOS_DIR}")

---
## Parte 4 · O juiz

Um modelo de fronteira (Gemini, pela API) recebe a pergunta, o gabarito (com o critério do que conta como certo) e a resposta, e dá o veredito: **correta**, **parcial** ou **incorreta**, com a justificativa. Ele tem acesso **somente leitura** às mesmas ferramentas do servidor MCP, para conferir o que o gabarito não resolve sozinho (por exemplo, um número da resposta que não está no gabarito), com no máximo 6 consultas por julgamento.

No sistema 3, quando o verificador devolveu a resposta, o juiz julga também a **primeira** resposta do agente (linha `anterior`): é o que mostra quantas devoluções mudaram o veredito.

Grava `juiz.csv` (veredito, justificativa, consultas e custo do juiz) e `juiz.jsonl` (com cada consulta do juiz), também a cada linha. Em erro de limite da API (429), espera e tenta de novo; se a cota do dia acabar, a célula para, e rodar de novo depois continua de onde parou. `PAUSA` espaça os julgamentos para respeitar o limite por minuto da camada gratuita.

In [ ]:
# @title Rodar o juiz
MODELO_JUIZ = "gemini-flash-latest"  # @param {type:"string"}
PAUSA = 10  # @param {type:"number"}

from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

import avaliacao.juiz as jz
jz.MODELO_JUIZ, jz.PAUSA = MODELO_JUIZ, PAUSA
JUIZ = jz.Juiz(SRV)

def progresso_juiz(l):
    marca = {"correta": "✅", "parcial": "🟡", "incorreta": "❌"}.get(l["veredito"], "⚠️")
    print(f"{marca} {l['execucao']:<16} {l['veredito']:<9} · {l['consultas_juiz']} consulta(s) · "
          f"{l['segundos_juiz']:>5.1f} s · {l['justificativa'][:150]}" + (f" · 💥 {l['erro']}" if l["erro"] else ""))

n = await jz.julgar_execucoes(JUIZ, RESULTADOS_DIR, mostrar=progresso_juiz)
print(f"\n{n} julgamentos agora · modelo {JUIZ.modelo}")

---
## Parte 5 · Resumo e ambiente

Um resumo rápido para conferir a rodada (a tabela completa, os gráficos e a análise de erros ficam no notebook 04). Acerto do juiz = veredito **correta**; parcial conta como erro. A última célula grava o ambiente (hardware, modelo, quantização e versões) em `ambiente.json`, ao lado dos resultados.

In [ ]:
# @title Resumo por sistema
import pandas as pd

ex = pd.read_csv(os.path.join(RESULTADOS_DIR, "execucoes.csv"))
jj = pd.read_csv(os.path.join(RESULTADOS_DIR, "juiz.csv"))
final = jj[jj["qual"] == "final"][["execucao", "veredito"]]
df = ex.merge(final, on="execucao", how="left")
df["acerto_juiz"] = df["veredito"] == "correta"

p95 = lambda s: s.quantile(0.95)
resumo = df.groupby("sistema", sort=False).agg(
    execucoes=("execucao", "count"), acerto_juiz=("acerto_juiz", "mean"), checagem=("checagem_ok", "mean"),
    trajetoria=("trajetoria_ok", "mean"), latencia_media=("segundos", "mean"), latencia_mediana=("segundos", "median"),
    latencia_p95=("segundos", p95), tokens_entrada=("tokens_entrada", "mean"), tokens_saida=("tokens_saida", "mean"),
    chamadas_modelo=("chamadas_modelo", "mean"), chamadas_ferramenta=("chamadas_ferramenta", "mean"),
    gpu_media=("gpu_medio", "mean"), ram_max_gb=("ram_max_gb", "max")).round(2)
display(resumo)

print("vereditos do juiz por sistema:")
display(pd.crosstab(df["sistema"], df["veredito"].fillna("sem julgamento")))

ant = jj[jj["qual"] == "anterior"].assign(execucao=lambda d: d["execucao"].str.removesuffix("-anterior"))
if len(ant):
    mud = ant.merge(final, on="execucao", suffixes=("_anterior", "_final"))
    print(f"sistema 3: {len(mud)} resposta(s) devolvida(s) pelo verificador; o veredito mudou em "
          f"{(mud['veredito_anterior'] != mud['veredito_final']).sum()}")
    display(mud[["execucao", "veredito_anterior", "veredito_final"]])

In [ ]:
# @title Ambiente (gravado em ambiente.json)
import json, platform
from importlib.metadata import version
import requests

info = requests.post("http://localhost:11434/api/show", json={"model": os.environ["AIRBNB_MODELO"]}).json()["details"]
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
AMBIENTE = {
    "commit": COMMIT, "python": platform.python_version(), "gpu": gpu, "cpus": os.cpu_count(),
    "ram_gb": round(__import__("psutil").virtual_memory().total / 2**30, 1),
    "ollama": requests.get("http://localhost:11434/api/version").json()["version"],
    "modelo": os.environ["AIRBNB_MODELO"], "parametros": info["parameter_size"], "quantizacao": info["quantization_level"],
    "juiz": JUIZ.modelo if "JUIZ" in globals() else None,
    "bibliotecas": {lib: version(lib) for lib in ["langgraph", "langchain-core", "langchain-ollama",
                                                  "langchain-mcp-adapters", "langchain-google-genai", "mcp"]},
    "preparacao_s": TEMPOS,
}
with open(os.path.join(RESULTADOS_DIR, "ambiente.json"), "w", encoding="utf-8") as f:
    json.dump(AMBIENTE, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(AMBIENTE, ensure_ascii=False, indent=2, default=str))